In [ ]:
import subprocess, pathlib, urllib.request
from pathlib import Path
import os
import sys
import pandas as pd, torch

In [ ]:
REPO_URL  = 'https://github.com/sashaboriskin/runumbers_asr.git'
REPO_REF  = 'main'
WEIGHTS_URL = 'https://github.com/sashaboriskin/runumbers_asr/releases/download/v0.2.0/best.pt'

# Clone repo
REPO = pathlib.Path('/tmp/runumbers_asr')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO)])

# Install dependencies
subprocess.check_call(['pip', 'install', '-q', 'num2words', 'jiwer', 'soundfile', 'librosa'])

# Download weights
CKPT = pathlib.Path('/tmp/best.pt')
if not CKPT.exists():
    urllib.request.urlretrieve(WEIGHTS_URL, CKPT)
print(f'ckpt size: {CKPT.stat().st_size / 1e6:.1f} MB')

In [ ]:
COMP_INPUT = Path("/kaggle/input/competitions")
print("=== /kaggle/input/ ===")
print(os.listdir("/kaggle/input/"))

if COMP_INPUT.exists():
    print("\n=== competitions/ ===")
    for p in COMP_INPUT.iterdir():
        print(p)

TEST_ROOT = COMP_INPUT / "asr-2026-spoken-numbers-recognition-challenge"
if TEST_ROOT.exists():
    print(f"\n=== {TEST_ROOT} ===")
    print(os.listdir(TEST_ROOT))
else:
    print(f"\nERROR: {TEST_ROOT} not found!")
    print("Make sure to add the competition data via 'Add Data' -> 'Competition Data'")

In [ ]:
sys.path.insert(0, str(REPO / 'src'))
from asr.infer import run_on_csv, InferConfig

TEST_ROOT = Path('/kaggle/input/competitions/asr-2026-spoken-numbers-recognition-challenge')
OUT  = Path('/kaggle/working/submission.csv')

cfg = InferConfig(
    ckpt=CKPT,
    csv=TEST_ROOT / 'test.csv',
    audio_root=TEST_ROOT,
    out=OUT,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    beam_width=20,
)
df = run_on_csv(cfg)
df.head()

In [ ]:
df.to_csv('/kaggle/working/submission.csv', index=False)